# 02 · O algoritmo Húngaro por lote: exemplo didático

A cada instante de lote (múltiplos de 60 s) há pedidos pendentes e veículos elegíveis. O despacho **sequencial** (mais próximo FIFO) escolhe pedido a pedido; a **atribuição conjunta** escolhe o conjunto de pares que maximiza o número de atendidos e, entre eles, minimiza a soma das distâncias de aproximação.

Os valores abaixo são **hipotéticos** (os mesmos da Tabela do artigo); servem para ilustrar o mecanismo, não vêm dos dados do estudo.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "execute o notebook dentro do repositório"
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
SEED = 20261108
np.random.seed(SEED)

In [2]:
from itertools import permutations
from types import SimpleNamespace

from tcc_frota.assignment import dispatch as dp

costs = np.array([[2, 3, 9], [1, 8, 7], [6, 4, 5]], float)  # km: linhas = pedidos P1..P3, colunas = veículos V1..V3
reqs = [SimpleNamespace(rid=f"P{i + 1}") for i in range(3)]
cands = {f"P{i + 1}": sorted((costs[i, j], f"V{j + 1}") for j in range(3)) for i in range(3)}

seq = dp.nearest_fifo(0.0, reqs, cands)
joint = dp.hungarian(0.0, reqs, cands)
cost = lambda pairs: sum(costs[int(r[1:]) - 1, int(v[1:]) - 1] for r, v in pairs)
print("sequencial:", seq, cost(seq), "km")
print("húngaro   :", sorted(joint), cost(joint), "km")

sequencial: [('P1', 'V1'), ('P2', 'V3'), ('P3', 'V2')] 13.0 km
húngaro   : [('P1', 'V2'), ('P2', 'V1'), ('P3', 'V3')] 9.0 km


## Conferência por enumeração exaustiva
Com 3 pedidos e 3 veículos há 3! = 6 atribuições completas; enumeramos todas e confirmamos que o Húngaro encontra o mínimo.

In [3]:
best = min((sum(costs[i, p[i]] for i in range(3)), p) for p in permutations(range(3)))
print("ótimo por enumeração:", best)
assert cost(seq) == 13.0 and cost(joint) == 9.0 == best[0]
assert abs(100 * (cost(joint) - cost(seq)) / cost(seq) - (-30.769)) < 0.01  # −30,8% neste exemplo
print(f"redução no exemplo: {100 * (cost(joint) - cost(seq)) / cost(seq):.1f}%")

ótimo por enumeração: (np.float64(9.0), (1, 0, 2))
redução no exemplo: -30.8%


## Matriz retangular e a coluna "não atribuído"
Sem o veículo V3 só dois pedidos podem ser atendidos. O Húngaro escolhe qual pedido fica sem veículo sem deixar de atender o máximo.

In [4]:
cands2 = {f"P{i + 1}": sorted((costs[i, j], f"V{j + 1}") for j in range(2)) for i in range(3)}
seq2 = dp.nearest_fifo(0.0, reqs, cands2)
joint2 = dp.hungarian(0.0, reqs, cands2)
print("sequencial:", seq2, cost(seq2), "km | húngaro:", sorted(joint2), cost(joint2), "km")
assert len(seq2) == len(joint2) == 2 and cost(joint2) == 4.0 and cost(seq2) == 10.0

sequencial: [('P1', 'V1'), ('P2', 'V2')] 10.0 km | húngaro: [('P1', 'V2'), ('P2', 'V1')] 4.0 km


## Verificação contra enumeração em instâncias aleatórias pequenas (sementes fixas)

In [5]:
import itertools


def brute(c):
    n, m = c.shape
    for k in range(min(n, m), -1, -1):
        found = None
        for rows in itertools.combinations(range(n), k):
            for cols in itertools.permutations(range(m), k):
                if all(np.isfinite(c[r, q]) for r, q in zip(rows, cols)):
                    s = sum(c[r, q] for r, q in zip(rows, cols))
                    found = s if found is None else min(found, s)
        if found is not None:
            return k, found


rng = np.random.default_rng(SEED)
for t in range(40):
    n, m = rng.integers(1, 5), rng.integers(1, 5)
    c = np.where(rng.random((n, m)) < 0.3, np.inf, np.round(rng.uniform(0.1, 9, (n, m)), 2))
    reqs_t = [SimpleNamespace(rid=f"P{i + 1}") for i in range(n)]
    cands_t = {f"P{i + 1}": sorted((c[i, j], f"V{j + 1}") for j in range(m) if np.isfinite(c[i, j])) for i in range(n)}
    cands_t = {k: v for k, v in cands_t.items() if v}
    pairs = dp.hungarian(0.0, reqs_t, cands_t)
    k, s = brute(c)
    got = sum(c[int(r[1:]) - 1, int(v[1:]) - 1] for r, v in pairs)
    assert len(pairs) == k and abs(got - s) < 0.02 * max(k, 1), (t, pairs, k, s, got)
print("40 instâncias: o Húngaro coincide com a enumeração (nº de pares e custo, tolerância de 1 m por par, pelo custo inteiro em metros).")

40 instâncias: o Húngaro coincide com a enumeração (nº de pares e custo, tolerância de 1 m por par, pelo custo inteiro em metros).


**Alcance desta verificação.** Ela mostra que, *em cada lote*, a solução é ótima para a formulação lexicográfica. Isso **não** implica ótimo ao longo do dia: a atribuição altera as posições futuras dos veículos e os pedidos futuros não são conhecidos.